In [1]:
using Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Nemo
include("fun.jl")

LDL (generic function with 1 method)

In [2]:
# [6,2,3], [11,42,3], [14,291,3], [14,9,5], [15,136,4], [17,58,5], [19, 7315, 3]

n,K,d = [19, 7315, 3]

R, x = polynomial_ring(QQ, :x)
N, z = number_field(x^2 - 3, :z)

name_file=string("cerf/pure/$n","_$K","_$d.jl")

include(name_file) # includes Y-> set of positive matrices, C -> Vector dual variable

print("done")

done

In [3]:
f4, size = f4_fun(n)
f2, size2 = f2_fun(n)

D = [sum([(2//1)^(-n)*K*Kw_field(j,i,n)*C[j+1] for j in 0:d-1]) for i in 0:n] # Define D

y=yijtp_fun_field(Y, n, f2, z) # Define yijtp

# Obj. funct

obj_fun = (D[1]-C[1]) - y[f4[0, 0, 0, 0]+1]/gamma_field(0,0,0,0,n,z)

println("Objective function = $obj_fun")
println("\n")

# Define Q

Q=[]

for i in d:n
    qu=-D[i+1]+(2*y[f4[i, 0, 0, 0]+1]+y[f4[i, i, i, i]+1])/gamma_field(i,0,0,0,n,z)
    
    push!(Q,qu/((2^n//K)-2))
end

# Constraint
        
ref=0 

for (i, j, t, p) in keys(f4)
    if (t - p) % 2 == 0 && i >= d && j >= d && i+j-t-p >= d
    
        total=0
        
        for (a, b, c, q) in keys(f4) 
            
            if t - p == c - q  && [i, j, i + j - t - p] in collect(permutations((a, b, a + b - c - q)))
    
                total += y[f4[a, b, c, q]+1]/gamma_field(a,b,c,q,n,z)+ Q[a+b-c-q-d+1]
            end
        end
    
        ref += total
        
    end
end

println("Sum of all constraints = $ref")


# All constraints should be 0 and the objective function 1

Objective function = 1


Sum of all constraints = 0


In [4]:
# Maximum and minimum eigenvalue of Y

Ypsd=[]

for k in 1:size2
    dim=length(Y[k][1,:])
    
    M=Matrix(undef,dim,dim)
    
    for i in 1:dim
        for j in 1:dim
            
            if typeof(Y[k][i,j])==AbsSimpleNumFieldElem
                
                M[i,j]=BigFloat(coeff(Y[k][i,j],0))+BigFloat(coeff(Y[k][i,j],1))*sqrt(big(3))
            else
                M[i,j]=Y[k][i,j]
               
        
            end
        end
    end
    
    push!(Ypsd,M)
    
end

ref_min=Inf
ref_max=0
for i in 1:size2
    mat=Matrix{BigFloat}(Ypsd[i])
    
    eigen=eigvals(mat)

    min=minimum(eigen)
    max=maximum(eigen)

    if ref_min > min
        ref_min=min
    end
    if max > ref_max
        ref_max=max
    end
end

println("Minimum eigenvalue = $ref_min\n")
println("Maximum eigenvalue = $ref_max")

Minimum eigenvalue = 107621.8549934665264024153728515011015935242643017561781807541022585829791605948

Maximum eigenvalue = 7.128098481324902630385371677505737286067093272969968598186659901543730998704249e+23


In [5]:
## Check positivity with LDL method

ref_min=Inf

upper=rationalize(big"1.732050807568877293529")
lower=rationalize(big"1.732050807568877293528")

for k in 1:size2
  
    A=Y[k]
    D=length(A[1,:])
    d=LDL(A,D)
    d_factor=Matrix(undef,D,1)
    
    for j in 1:D
        d[j]=d[j]*z^0
        if coeff(d[j],1) < 0
            factor=coeff(d[j],0)+upper*coeff(d[j],1) # bigger value of sqrt(3)
        else
            factor=coeff(d[j],0)+lower*coeff(d[j],1) # smaller value of sqrt(3)
        end

        d_factor[j]=factor

    end
    
    ref=minimum(d_factor)

    if ref_min > ref
        ref_min=ref
    end

end


ref_min_float=BigFloat(ref_min)

println("Minimum value (Fraction) = $ref_min")
println("Minimum value (Float) = $ref_min_float")

Minimum value (Fraction) = ...
Minimum value (Float) = 4.768279840512140500970681500978666238531533956371272805033710669647331899385304e+10
